# 🦀 Day 7: Mini-Project — Data Pipeline Processor

Process structured data using closures and iterator chains!

---

## 🔧 Step 1: Data Model

In [ ]:
use std::collections::HashMap;

#[derive(Debug, Clone)]
struct Record {
    fields: HashMap<String, String>,
}

impl Record {
    fn new(pairs: &[(&str, &str)]) -> Self {
        Record {
            fields: pairs.iter().map(|(k, v)| (k.to_string(), v.to_string())).collect()
        }
    }
    fn get(&self, key: &str) -> Option<&str> { self.fields.get(key).map(|s| s.as_str()) }
    fn get_f64(&self, key: &str) -> Option<f64> { self.get(key)?.parse().ok() }
}

fn parse_csv_records(csv: &str) -> Vec<Record> {
    let mut lines = csv.lines();
    let headers: Vec<&str> = lines.next().unwrap_or("").split(',').map(|s| s.trim()).collect();
    lines.map(|line| {
        let values: Vec<&str> = line.split(',').map(|s| s.trim()).collect();
        let pairs: Vec<(&str, &str)> = headers.iter().zip(values.iter()).map(|(&h, &v)| (h, v)).collect();
        Record::new(&pairs)
    }).collect()
}

println!("Data model ready! ✅");

## 🔗 Step 2: Pipeline Builder

In [ ]:
type TransformFn = Box<dyn Fn(Vec<Record>) -> Vec<Record>>;

struct Pipeline {
    steps: Vec<(String, TransformFn)>,
}

impl Pipeline {
    fn new() -> Self { Pipeline { steps: Vec::new() } }

    fn filter(mut self, name: &str, predicate: impl Fn(&Record) -> bool + 'static) -> Self {
        self.steps.push((name.to_string(), Box::new(move |records| {
            records.into_iter().filter(|r| predicate(r)).collect()
        })));
        self
    }

    fn map_field(mut self, name: &str, field: &str, transform: impl Fn(&str) -> String + 'static) -> Self {
        let field = field.to_string();
        self.steps.push((name.to_string(), Box::new(move |records| {
            records.into_iter().map(|mut r| {
                if let Some(val) = r.fields.get(&field).cloned() {
                    r.fields.insert(field.clone(), transform(&val));
                }
                r
            }).collect()
        })));
        self
    }

    fn sort_by(mut self, name: &str, field: &str, ascending: bool) -> Self {
        let field = field.to_string();
        self.steps.push((name.to_string(), Box::new(move |mut records| {
            records.sort_by(|a, b| {
                let va = a.get(&field).unwrap_or("");
                let vb = b.get(&field).unwrap_or("");
                let cmp = va.partial_cmp(vb).unwrap_or(std::cmp::Ordering::Equal);
                if ascending { cmp } else { cmp.reverse() }
            });
            records
        })));
        self
    }

    fn execute(self, data: Vec<Record>) -> Vec<Record> {
        let mut result = data;
        for (name, step) in &self.steps {
            let before = result.len();
            result = step(result);
            println!("  📌 {} — {} → {} records", name, before, result.len());
        }
        result
    }
}

println!("Pipeline builder ready! ✅");

## 🧪 Step 3: Run It!

In [ ]:
let csv_data = "name, age, salary, department
Alice, 30, 75000, Engineering
Bob, 25, 55000, Marketing
Charlie, 35, 95000, Engineering
Diana, 28, 62000, Design
Eve, 32, 88000, Engineering
Frank, 27, 48000, Marketing
Grace, 31, 71000, Design";

let records = parse_csv_records(csv_data);
println!("Loaded {} records\n", records.len());

// Build and execute pipeline
println!("🔗 Running pipeline:");
let results = Pipeline::new()
    .filter("Age > 26", |r| r.get_f64("age").unwrap_or(0.0) > 26.0)
    .filter("Salary > 60000", |r| r.get_f64("salary").unwrap_or(0.0) > 60000.0)
    .map_field("Uppercase names", "name", |s| s.to_uppercase())
    .sort_by("Sort by salary", "salary", false)
    .execute(records);

println!("\n📊 Results:");
for r in &results {
    println!("  {} | age {} | ${} | {}",
        r.get("name").unwrap_or("?"),
        r.get("age").unwrap_or("?"),
        r.get("salary").unwrap_or("?"),
        r.get("department").unwrap_or("?"),
    );
}

In [ ]:
// Aggregations using iterator chains
let csv_data = "name, age, salary, department
Alice, 30, 75000, Engineering
Bob, 25, 55000, Marketing
Charlie, 35, 95000, Engineering
Diana, 28, 62000, Design
Eve, 32, 88000, Engineering
Frank, 27, 48000, Marketing
Grace, 31, 71000, Design";

let records = parse_csv_records(csv_data);

// Group by department
let mut by_dept: HashMap<String, Vec<&Record>> = HashMap::new();
for r in &records {
    let dept = r.get("department").unwrap_or("Unknown").to_string();
    by_dept.entry(dept).or_default().push(r);
}

println!("📊 Department Summary:");
for (dept, members) in &by_dept {
    let salaries: Vec<f64> = members.iter()
        .filter_map(|r| r.get_f64("salary"))
        .collect();
    let avg = salaries.iter().sum::<f64>() / salaries.len() as f64;
    let max = salaries.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
    println!("  {} — {} people, avg ${:.0}, max ${:.0}", dept, members.len(), avg, max);
}

## 🎯 Key Takeaways

1. **Builder pattern** + closures = flexible, composable pipelines
2. `Box<dyn Fn>` stores different closures in one collection
3. Iterator chains handle aggregation, grouping, and transformation
4. Each pipeline step is independently testable
5. Closures capture configuration, making steps reusable

---
**Next Week:** Smart Pointers & Memory! 🧠